### TD 1 · Représenter : du modèle linéaire au réseau de neurones

In [1]:
import sys
from math import comb
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from src.data import SEED, set_seed

set_seed(SEED)
# Des décimales lisibles plutôt que la notation scientifique : tout le TD consiste
# à comparer des tableaux affichés avec des calculs faits sur papier.
np.set_printoptions(precision=6, suppress=True)
# Sur quatre points, MLPClassifier signale parfois une convergence incomplète. Ce
# message n'apporte rien ici ; on le masque pour garder les sorties lisibles.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
print("prêt, seed", SEED)

ModuleNotFoundError: No module named 'matplotlib'

### Exercice 1.1 · XOR et logistic regression

In [ ]:
# XOR : les deux points de classe 1 sont sur une diagonale, les deux points de
# classe 0 sur l'autre.
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])

lineaire = LogisticRegression().fit(X_xor, y_xor)
print("Logistic regression sur XOR")
print(f"  prédictions  : {lineaire.predict(X_xor)}")
print(f"  vérité       : {y_xor}")
print(f"  accuracy     : {lineaire.score(X_xor, y_xor):.2f}")
# Les quatre probabilités à 0,5 : le modèle n'a aucune opinion.
print(f"  probabilités : {lineaire.predict_proba(X_xor)[:, 1].round(4)}")

l'accuracy du modèle est de 50% donc le modèle n'est pas trop sur.

TODO : écrivez exactitude_droite(w1, w2, b), qui renvoie la proportion de points de XOR bien classés par la règle « classe 1 si w1*x1 + w2*x2 + b > 0 ».
Essayez ensuite au moins trois droites de votre choix.

In [ ]:
def exactitude_droite(w1, w2, b):
    # La règle d'une droite : un côté prédit 1, l'autre prédit 0.
    predit = (X_xor @ np.array([w1, w2]) + b > 0).astype(int)
    return float((predit == y_xor).mean())


for w1, w2, b in [(1, 1, -0.5), (1, 1, -1.5), (1, -1, -0.5)]:
    print(f"  droite w1={w1:+}, w2={w2:+}, b={b:+}  ->  accuracy "
          f"{exactitude_droite(w1, w2, b):.2f}")

In [ ]:
# Recherche exhaustive sur une grille de 41 x 41 x 41 droites (près de 69 000).
# Si une droite pouvait tout séparer, on la trouverait ici.
grille = np.linspace(-2, 2, 41)
meilleure = max(exactitude_droite(a, b, c)
                for a in grille for b in grille for c in grille)
print(f"meilleure accuracy d'une droite sur la grille : {meilleure:.2f}")

# Exercice 1.2 · Réparer XOR à la main

In [ ]:
# TODO : construisez X_plus = [x1, x2, x1*x2] avec np.column_stack, ajustez
# LogisticRegression(C=100) et affichez l'accuracy, les probabilités et les
# coefficients. (C=100 signifie une pénalité faible : voir l'analyse.)
# On fabrique la feature qui manque : x1*x2 ne vaut 1 que pour le point (1, 1).
X_plus = np.column_stack([X_xor, X_xor[:, 0] * X_xor[:, 1]])
# C est l'inverse de la force de la pénalité (séance 5). Avec C=1 par défaut, quatre
# points ne suffisent pas à vaincre la régularisation et l'accuracy reste à 0,75.
avec_produit = LogisticRegression(C=100).fit(X_plus, y_xor)
print(f"  accuracy     : {avec_produit.score(X_plus, y_xor):.2f}")
print(f"  probabilités : {avec_produit.predict_proba(X_plus)[:, 1].round(3)}")
print(f"  coefficients : {avec_produit.coef_[0].round(2)}   "
      f"biais {avec_produit.intercept_[0]:.2f}")

In [ ]:
# Le coût de la méthode manuelle : le nombre de produits à envisager explose.
for d in (2, 10, 50):
    print(f"  {d:3d} features : {comb(d, 2):6d} produits de deux, "
          f"{comb(d, 3):7d} produits de trois")

# Exercice 1.3 · Un neurone est une logistic regression

In [ ]:
def sigmoide(z):
    # Écrase n'importe quel réel dans (0, 1) : la sortie se lit comme une probabilité.
    return 1.0 / (1.0 + np.exp(-z))

L'interprète c'est le point de Y quand X = 0

In [ ]:
# TODO : écrivez neurone(X, w, b, g) = g(X @ w + b). Puis, avec g = sigmoide et les
# coefficients de avec_produit (coef_[0] et intercept_[0]), comparez vos
# probabilités à avec_produit.predict_proba(X_plus)[:, 1].

def neurone(X, w, b, g):
    # Une somme pondérée, un biais, une activation function. Rien d'autre.
    return g(X @ w + b)


p_neurone = neurone(X_plus, avec_produit.coef_[0], avec_produit.intercept_[0], sigmoide)
p_sklearn = avec_produit.predict_proba(X_plus)[:, 1]
print(f"  neurone : {p_neurone.round(6)}")
print(f"  sklearn : {p_sklearn.round(6)}")

In [ ]:
# Vérification : les deux calculs doivent coïncider à la précision machine.
ecart = np.abs(np.asarray(p_neurone) - np.asarray(p_sklearn)).max()
print(f"  écart maximal : {ecart:.1e}   {'OK' if ecart < 1e-12 else 'à revoir'}")

# Exercice 1.4 · Activations et dérivées

**Explication.** Quatre activations courantes :

| nom | g(z) | g'(z) |
|---|---|---|
| sigmoïde | 1 / (1 + e⁻ᶻ) | g(z)·(1 - g(z)) |
| tanh | tanh z | 1 - tanh² z |
| ReLU (*rectified linear unit*, unité linéaire rectifiée) | max(0, z) | 1 si z > 0, sinon 0 |
| Leaky ReLU (ReLU « qui fuit ») | z si z > 0, sinon α·z | 1 si z > 0, sinon α |

Pourquoi s'intéresser aux **dérivées** ? Parce que la backpropagation (TD 2)
multiplie le gradient par g'(z) à chaque couche qu'il traverse.

In [ ]:
# TODO : complétez les sept fonctions. np.tanh, np.maximum et np.where suffisent.


def d_sigmoide(z):
    s = sigmoide(z)
    return s * (1 - s)


def tanh(z):
    return np.tanh(z)


def d_tanh(z):
    return 1 - np.tanh(z) ** 2


def relu(z):
    return np.maximum(0.0, z)


def d_relu(z):
    # La dérivée vaut 1 ou 0 : le gradient passe tel quel, ou pas du tout.
    return (z > 0).astype(float)


def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)


def d_leaky_relu(z, alpha=0.1):
    # Jamais nulle : une unité « éteinte » reçoit encore un peu de gradient.
    return np.where(z > 0, 1.0, alpha)

In [ ]:
# Vérification par finite differences (voir TD 2) : on compare chaque dérivée codée
# à la pente mesurée numériquement. On évite z proche de 0, où ReLU n'est pas
# dérivable.
z_test = np.random.default_rng(0).uniform(-4, 4, 200)
z_test = z_test[np.abs(z_test) > 1e-3]
h = 1e-6
for nom, g, dg in [("sigmoïde", sigmoide, d_sigmoide), ("tanh", tanh, d_tanh),
                   ("ReLU", relu, d_relu), ("Leaky ReLU", leaky_relu, d_leaky_relu)]:
    try:
        pente = (g(z_test + h) - g(z_test - h)) / (2 * h)
        ecart = np.abs(pente - dg(z_test)).max()
        print(f"  {nom:10s} écart max {ecart:.1e}   {'OK' if ecart < 1e-6 else 'à revoir'}")
    except NotImplementedError:
        print(f"  {nom:10s} à compléter")

In [ ]:
z = np.linspace(-5, 5, 401)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for nom, g, dg in [("sigmoïde", sigmoide, d_sigmoide), ("tanh", tanh, d_tanh),
                   ("ReLU", relu, d_relu), ("Leaky ReLU", leaky_relu, d_leaky_relu)]:
    axes[0].plot(z, g(z), lw=2, label=nom)
    axes[1].plot(z, dg(z), lw=2, label=nom)
axes[0].set(title="g(z)", xlabel="z", ylim=(-1.3, 3))
axes[1].set(title="g'(z) : le facteur appliqué au gradient", xlabel="z")
axes[1].axhline(0.25, color="grey", ls=":")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TODO :
#  (a) valeur maximale de d_sigmoide sur np.linspace(-6, 6, 1201) ;
#  (b) ce maximum élevé à la puissance 10 (dix couches sigmoïdes, dans le meilleur cas) ;
#  (c) la même chose si chaque unité est à z = 2 (déjà un peu saturée) ;
#  (d) la même chose pour ReLU avec des unités actives (z = 2).

maximum = d_sigmoide(np.linspace(-6, 6, 1201)).max()
print(f"  (a) dérivée maximale de la sigmoïde : {maximum:.4f}")
print(f"  (b) sur 10 couches, au mieux        : {maximum ** 10:.2e}")
print(f"  (c) sur 10 couches à z = 2          : {d_sigmoide(2.0) ** 10:.2e}")
print(f"  (d) ReLU, 10 couches actives        : {d_relu(np.array(2.0)) ** 10:.1f}")

# Exercice 1.5 · Empiler des couches linéaires

In [ ]:
# TODO :
#  1. avec rng = np.random.default_rng(1), tirez W1 (4, 2), b1 (4,), W2 (1, 4), b2 (1,) ;
#  2. tirez 100 entrées X_alea de forme (100, 2) ;
#  3. calculez deux_couches = (X_alea @ W1.T + b1) @ W2.T + b2 (aucune activation) ;
#  4. calculez W = W2 @ W1, b = W2 @ b1 + b2, puis une_couche = X_alea @ W.T + b ;
#  5. affichez l'écart maximal entre deux_couches et une_couche.

rng = np.random.default_rng(1)
W1_l, b1_l = rng.normal(size=(4, 2)), rng.normal(size=4)
W2_l, b2_l = rng.normal(size=(1, 4)), rng.normal(size=1)
X_alea = rng.normal(size=(100, 2))

deux_couches = (X_alea @ W1_l.T + b1_l) @ W2_l.T + b2_l
# La couche équivalente, calculée une fois pour toutes.
W_eq, b_eq = W2_l @ W1_l, W2_l @ b1_l + b2_l
une_couche = X_alea @ W_eq.T + b_eq
print(f"  écart maximal, 2 couches contre 1 : {np.abs(deux_couches - une_couche).max():.1e}")

# Même chose avec dix couches de largeurs diverses : le produit des matrices suffit.
largeurs = [2, 8, 16, 8, 8, 4, 8, 16, 8, 4, 1]
couches = [(rng.normal(size=(m, n)), rng.normal(size=m))
           for n, m in zip(largeurs[:-1], largeurs[1:])]
sortie = X_alea
W_tot, b_tot = np.eye(2), np.zeros(2)
for W, b in couches:
    sortie = sortie @ W.T + b
    W_tot, b_tot = W @ W_tot, W @ b_tot + b
print(f"  écart maximal, 10 couches contre 1 : "
      f"{np.abs(sortie - (X_alea @ W_tot.T + b_tot)).max():.1e}  "
      f"(relatif {np.abs(sortie - (X_alea @ W_tot.T + b_tot)).max() / np.abs(sortie).max():.1e})")

In [ ]:
# TODO : pour 5 seeds (random_state=0..4), entraînez sur XOR :
#   A. MLPClassifier(hidden_layer_sizes=(8,) * 10, activation="identity", solver="lbfgs", max_iter=2000)
#   B. MLPClassifier(hidden_layer_sizes=(4,), activation="tanh", solver="lbfgs", max_iter=2000)
#   C. MLPClassifier(hidden_layer_sizes=(4,), activation="relu", solver="lbfgs", max_iter=2000)
# et affichez les 5 valeurs d'accuracy de chacun.

reseaux = {
    "A. 10 couches, identité": dict(hidden_layer_sizes=(8,) * 10, activation="identity"),
    "B. 1 couche de 4, tanh": dict(hidden_layer_sizes=(4,), activation="tanh"),
    "C. 1 couche de 4, ReLU": dict(hidden_layer_sizes=(4,), activation="relu"),
}
for nom, params in reseaux.items():
    exactitudes = [MLPClassifier(solver="lbfgs", max_iter=2000, random_state=g, **params)
                   .fit(X_xor, y_xor).score(X_xor, y_xor) for g in range(5)]
    print(f"  {nom:26s} {exactitudes}")

# Exercice 1.6 · Le forward pass à la main

In [ ]:
x = np.array([1.0, 0.0])
y_vrai = 1.0
W1 = np.array([[0.5, -0.5],
               [0.5, 0.5]])
b1 = np.array([0.0, 0.1])
W2 = np.array([[1.0, -1.0]])
b2 = np.array([0.2])

In [ ]:
# TODO (1) : sans rien calculer, donnez la forme (un tuple) de chaque quantité.
formes = {"W1": (2, 2), "b1": (2,), "z1": (2,), "a1": (2,),
          "W2": (1, 2), "b2": (1,), "z2": (1,), "a2": (1,)}
# z1 = [0,5*1 + (-0,5)*0 + 0 ; 0,5*1 + 0,5*0 + 0,1] ; les deux sont positifs, ReLU
# les laisse passer ; z2 = 1*0,5 - 1*0,6 + 0,2.
z1_main, a1_main, z2_main = [0.5, 0.6], [0.5, 0.6], 0.1
a2_main, perte_main = 0.525, 0.644

In [ ]:
# Vérification : le calcul de référence, fait par NumPy.
z1 = W1 @ x + b1
a1 = np.maximum(0.0, z1)
z2 = W2 @ a1 + b2
a2 = sigmoide(z2)
perte = float(-np.log(a2[0]))
reference = {"W1": W1, "b1": b1, "z1": z1, "a1": a1, "W2": W2, "b2": b2, "z2": z2, "a2": a2}


def verifier(nom, valeur, attendu, tol=2e-3):
    if valeur is None:
        print(f"  {nom:6s} : à compléter")
    elif np.allclose(np.ravel(valeur), np.ravel(attendu), atol=tol):
        print(f"  {nom:6s} : OK")
    else:
        print(f"  {nom:6s} : à revoir (vous : {valeur}, attendu : {np.round(attendu, 6)})")


print("Formes")
for nom, forme in formes.items():
    etat = "à compléter" if forme is None else (
        "OK" if tuple(forme) == reference[nom].shape else f"à revoir ({reference[nom].shape})")
    print(f"  {nom:6s} : {etat}")
print("Valeurs")
for nom, valeur, attendu in [("z1", z1_main, z1), ("a1", a1_main, a1), ("z2", z2_main, z2),
                             ("a2", a2_main, a2), ("loss", perte_main, perte)]:
    verifier(nom, valeur, attendu)
print(f"\nréférence : a2 = {a2[0]:.6f}, loss = {perte:.6f}")

# Exercice 1.7 · Universal approximation, par construction

In [ ]:
xs = np.linspace(-3, 3, 7)
print("  x                   :", xs)
print("  relu(x) + relu(-x)  :", np.maximum(0, xs) + np.maximum(0, -xs))
print("  |x|                 :", np.abs(xs))

In [ ]:
# TODO : complétez reseau_charnieres.
def reseau_charnieres(x, noeuds, cible):
    """Approche `cible` par une somme de ReLU, une unité par point de rupture."""
    valeurs = cible(noeuds)
    pentes = np.diff(valeurs) / np.diff(noeuds)
    # Chaque unité change la pente au moment où elle s'allume.
    coefs = np.r_[pentes[0], np.diff(pentes)]
    x = np.asarray(x, dtype=float)
    return valeurs[0] + sum(c * np.maximum(0.0, x - t) for c, t in zip(coefs, noeuds[:-1]))

In [ ]:
# Erreur maximale sur [0, 2π] quand on double le nombre d'unités.
x_fin = np.linspace(0, 2 * np.pi, 4001)
for k in (3, 6, 12, 24, 48):
    noeuds = np.linspace(0, 2 * np.pi, k + 1)
    erreur = np.abs(reseau_charnieres(x_fin, noeuds, np.sin) - np.sin(x_fin)).max()
    print(f"  {k:3d} unités : erreur maximale {erreur:.4f}")

# Hors de la zone : on évalue le réseau à 12 unités sur [2π, 3π].
noeuds12 = np.linspace(0, 2 * np.pi, 13)
x_hors = np.linspace(2 * np.pi, 3 * np.pi, 500)
erreur_hors = np.abs(reseau_charnieres(x_hors, noeuds12, np.sin) - np.sin(x_hors)).max()
print(f"\n  12 unités, hors de la zone [2π, 3π] : erreur maximale {erreur_hors:.2f}")

In [ ]:
# TODO : déplacez les noeuds intérieurs (gardez 0 et 2π aux extrémités, 9 noeuds).
# Idée : resserrer les noeuds là où la courbure |sin''| = |sin| est forte (vers π/2
# et 3π/2) et les espacer là où la courbe est presque droite (vers 0, π, 2π). On
# répartit les noeuds selon une densité proportionnelle à la racine de la courbure.
grille_fine = np.linspace(0, 2 * np.pi, 10001)
densite = np.sqrt(np.abs(np.sin(grille_fine))) + 0.05
cumul = np.cumsum(densite)
cumul = (cumul - cumul[0]) / (cumul[-1] - cumul[0])
noeuds_perso = np.interp(np.linspace(0, 1, 9), cumul, grille_fine)
erreur_reguliere = np.abs(reseau_charnieres(x_fin, np.linspace(0, 2 * np.pi, 9), np.sin)
                          - np.sin(x_fin)).max()
erreur_perso = np.abs(reseau_charnieres(x_fin, noeuds_perso, np.sin) - np.sin(x_fin)).max()
print(f"  noeuds choisis : {noeuds_perso.round(2)}")
print(f"  espacement régulier : {erreur_reguliere:.4f}   vos noeuds : {erreur_perso:.4f}")

### TD 2 · Apprendre : la backpropagation, du papier à PyTorch

In [ ]:
from pathlib import Path
print(Path.cwd())

In [ ]:
!ls

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from src.data import SEED, set_seed

set_seed(SEED)
np.set_printoptions(precision=6, suppress=True)
# Quatre threads : sur des tenseurs minuscules, davantage de threads ne fait que
# ralentir (voir TD 3, où ce réglage est expliqué en détail).
torch.set_num_threads(4)

# Les quatre points de XOR, qui serviront à partir de l'exercice 2.3.
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0.0, 1.0, 1.0, 0.0])


def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


# La fiche de la séance 8 : un exemple, neuf paramètres fixés à la main.
x = np.array([1.0, 0.0])
y_vrai = 1.0
PARAMS = {"W1": np.array([[0.5, -0.5], [0.5, 0.5]]), "b1": np.array([0.0, 0.1]),
          "W2": np.array([[1.0, -1.0]]), "b2": np.array([0.2])}


def perte_fiche(p):
    """Forward pass de la fiche, vu comme une fonction des paramètres."""
    a1 = np.maximum(0.0, p["W1"] @ x + p["b1"])
    a2 = sigmoide(p["W2"] @ a1 + p["b2"])
    return float(-(y_vrai * np.log(a2[0]) + (1 - y_vrai) * np.log(1 - a2[0])))


print(f"loss de la fiche au départ : {perte_fiche(PARAMS):.6f}")

In [ ]:
f = lambda w: w ** 3  # noqa: E731
for eps in (1e-1, 1e-3, 1e-5):
    print(f"  eps = {eps:.0e} : pente mesurée {(f(2 + eps) - f(2 - eps)) / (2 * eps):.10f}")

In [ ]:
def gradient_numerique(fonction, theta, eps=1e-6):
    """Gradient de `fonction` en `theta`, case par case, par différence centrée."""
    grad = np.zeros_like(theta, dtype=float)
    for i in np.ndindex(theta.shape):
        plus, moins = theta.astype(float).copy(), theta.astype(float).copy()
        plus[i] += eps
        moins[i] -= eps
        grad[i] = (fonction(plus) - fonction(moins)) / (2 * eps)
    return grad

In [ ]:
theta_s = np.array([0.3, 1.2, -2.0])
pas = np.logspace(-1, -13, 13)
erreurs = [np.abs(gradient_numerique(lambda t: np.sum(np.sin(t)), theta_s, eps)
                  - np.cos(theta_s)).max() for eps in pas]
for eps, err in zip(pas, erreurs):
    print(f"  eps = {eps:.0e} : erreur {err:.1e}")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.loglog(pas, erreurs, "o-")
ax.set(xlabel="eps", ylabel="erreur maximale", title="Erreur des finite differences")
ax.invert_xaxis()
plt.tight_layout()
plt.show()

In [ ]:
# TODO : pour f(θ) = somme de sin(θ), dont le gradient exact est cos(θ), mesurez l'erreur
# maximale de gradient_numerique pour eps = 1e-1, 1e-2, ..., 1e-13 (np.logspace), en
# theta = np.array([0.3, 1.2, -2.0]). Tracez l'erreur en fonction de eps (échelles log).

theta_s = np.array([0.3, 1.2, -2.0])
pas = np.logspace(-1, -13, 13)
erreurs = [np.abs(gradient_numerique(lambda t: np.sum(np.sin(t)), theta_s, eps)
                  - np.cos(theta_s)).max() for eps in pas]
for eps, err in zip(pas, erreurs):
    print(f"  eps = {eps:.0e} : erreur {err:.1e}")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.loglog(pas, erreurs, "o-")
ax.set(xlabel="eps", ylabel="erreur maximale", title="Erreur des finite differences")
ax.invert_xaxis()
plt.tight_layout()
plt.show()


### Exercice 2.2 · La backpropagation à la main

In [ ]:
# TODO : complétez le backward pass. Le forward pass est donné.

W1, b1, W2, b2 = PARAMS["W1"], PARAMS["b1"], PARAMS["W2"], PARAMS["b2"]
z1 = W1 @ x + b1
a1 = np.maximum(0.0, z1)
z2 = W2 @ a1 + b2
a2 = sigmoide(z2)

dz2 = a2 - y_vrai                # (1,)   prédiction moins vérité
dW2 = np.outer(dz2, a1)          # (1, 2) activation entrante x erreur sortante
db2 = dz2                        # (1,)   l'entrée d'un biais vaut 1
da1 = W2.T @ dz2                 # (2,)   l'erreur remonte par les poids
dz1 = da1 * (z1 > 0)             # (2,)   ReLU laisse passer, ou coupe
dW1 = np.outer(dz1, x)           # (2, 2)
db1 = dz1                        # (2,)

In [ ]:
# Vérification par l'arbitre de l'exercice 2.1 : chaque gradient analytique est comparé
# à la pente mesurée en poussant le paramètre correspondant.
analytiques = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
for nom, grad in analytiques.items():
    def perte_selon(valeur, nom=nom):
        return perte_fiche({**PARAMS, nom: valeur})
    mesure = gradient_numerique(perte_selon, PARAMS[nom])
    if grad is None:
        print(f"  dL/d{nom:3s} : à compléter")
    else:
        ecart = np.abs(np.asarray(grad) - mesure).max()
        print(f"  dL/d{nom:3s} = {np.round(np.asarray(grad), 6).tolist()}   "
              f"écart {ecart:.1e}   {'OK' if ecart < 1e-7 else 'à revoir'}")

In [ ]:
# Un step de gradient descent, learning rate 0,1 : la loss doit baisser.
TAUX = 0.1
if any(g is None for g in analytiques.values()):
    print("  complétez d'abord le backward pass")
else:
    nouveaux = {nom: PARAMS[nom] - TAUX * np.asarray(analytiques[nom]) for nom in PARAMS}
    print(f"  loss avant le step : {perte_fiche(PARAMS):.6f}")
    print(f"  loss après le step : {perte_fiche(nouveaux):.6f}")

### Exercice 2.3 · Le réseau NumPy, par batch

In [ ]:
# TODO : complétez forward, backward et step. L'initialisation, la loss et
# l'activation sont données.


class ReseauNumPy:
    """Linéaire -> activation -> linéaire -> sigmoïde, cross-entropy binaire."""

    def __init__(self, n_entrees=2, n_cachees=4, graine=SEED):
        rng = np.random.default_rng(graine)
        # Initialisation de He : écart-type sqrt(2 / nombre d'entrées de la couche).
        self.W1 = rng.normal(0, np.sqrt(2 / n_entrees), (n_entrees, n_cachees))
        self.b1 = np.zeros(n_cachees)
        self.W2 = rng.normal(0, np.sqrt(2 / n_cachees), (n_cachees, 1))
        self.b2 = np.zeros(1)

    def activation(self, Z):
        return np.maximum(0.0, Z)

    def d_activation(self, Z):
        return (Z > 0).astype(float)

    def forward(self, X):
        # Tout est gardé sur self : le backward pass en a besoin. C'est exactement ce
        # que fait le graphe de PyTorch.
        self.Z1 = X @ self.W1 + self.b1              # (n, h)
        self.A1 = self.activation(self.Z1)           # (n, h)
        self.Z2 = self.A1 @ self.W2 + self.b2        # (n, 1)
        self.A2 = sigmoide(self.Z2)                  # (n, 1)
        return self.A2

    def perte(self, y):
        p = np.clip(self.A2[:, 0], 1e-12, 1 - 1e-12)
        return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

    def backward(self, X, y):
        n = len(y)
        dZ2 = (self.A2 - y.reshape(-1, 1)) / n       # (n, 1)  a2 - y, moyenné
        dW2 = self.A1.T @ dZ2                        # (h, 1)  somme des produits extérieurs
        db2 = dZ2.sum(axis=0)                        # (1,)
        dA1 = dZ2 @ self.W2.T                        # (n, h)
        dZ1 = dA1 * self.d_activation(self.Z1)       # (n, h)
        dW1 = X.T @ dZ1                              # (2, h)
        db1 = dZ1.sum(axis=0)                        # (h,)
        return dW1, db1, dW2, db2

    def step(self, grads, taux):
        dW1, db1, dW2, db2 = grads
        self.W1 -= taux * dW1
        self.b1 -= taux * db1
        self.W2 -= taux * dW2
        self.b2 -= taux * db2

In [ ]:
# Vérification : les gradients de backward() contre l'arbitre, sur les quatre points.
# Les biais cachés sont décalés à 0,05 pour la vérification : voir la remarque
# ci-dessous.
DECALAGE = 0.05

#on a cree un objet de la classe (r)

def reseau_de_test():
    r = ReseauNumPy()
    r.b1[:] = DECALAGE
    return r


reseau = reseau_de_test()
reseau.forward(X_xor)
grads = dict(zip(["W1", "b1", "W2", "b2"], reseau.backward(X_xor, y_xor)))
for nom in grads:
    def perte_selon(valeur, nom=nom):
        copie = reseau_de_test()
        setattr(copie, nom, valeur)
        copie.forward(X_xor)
        return copie.perte(y_xor)
    ecart = np.abs(grads[nom] - gradient_numerique(perte_selon, getattr(reseau, nom))).max()
    print(f"  dL/d{nom:3s} : écart {ecart:.1e}   {'OK' if ecart < 1e-7 else 'à revoir'}")

In [ ]:
def entrainer(reseau, X, y, taux=0.5, epoques=4000):
    """Gradient descent sur le batch complet ; renvoie l'historique de la loss."""
    historique = []
    for _ in range(epoques):
        reseau.forward(X)
        historique.append(reseau.perte(y))
        reseau.step(reseau.backward(X, y), taux)
    return historique


reseau = ReseauNumPy()
historique = entrainer(reseau, X_xor, y_xor)
predictions = reseau.forward(X_xor)[:, 0]
print(f"  loss initiale  : {historique[0]:.4f}")
print(f"  loss finale    : {historique[-1]:.6f}")
print(f"  prédictions    : {predictions.round(4)}")
print(f"  accuracy       : {((predictions > 0.5) == y_xor).mean():.2f}")

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(historique)
ax.set(yscale="log", xlabel="epoch", ylabel="cross-entropy",
       title="XOR appris par un réseau NumPy à 4 unités cachées")
plt.tight_layout()
plt.show()

### Exercice 2.4 · Le symmetry breaking

In [ ]:
#Regarder la notion de symetrie breaking
for nom, valeur in [("aléatoire (He)", None), ("tout à zéro", 0.0), ("tout à 0,5", 0.5)]:
    r = ReseauNumPy()
    if valeur is not None:
        r.W1[:] = valeur
        r.W2[:] = valeur
    h = entrainer(r, X_xor, y_xor)
    exact = ((r.forward(X_xor)[:, 0] > 0.5) == y_xor).mean()
    distinctes = np.unique(r.W1.round(6), axis=1).shape[1]
    print(f"  {nom:15s} loss finale {h[-1]:.4f}   accuracy {exact:.2f}   "
          f"unités distinctes {distinctes}/4")

### Exercice 2.5 · De NumPy à PyTorch

In [ ]:
# TODO : écrivez la version PyTorch.
#   - torch.manual_seed(SEED) puis modele = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
#   - perte_fn = nn.BCEWithLogitsLoss() ; optimiseur = torch.optim.SGD(modele.parameters(), lr=0.5)
#   - Xt (float32) et yt de forme (4, 1)
#   - 4 000 epochs : zero_grad, forward pass, loss, backward, step
# Affichez la loss finale et les probabilités prédites (torch.sigmoid des logits).

torch.manual_seed(SEED)
modele = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
perte_fn = nn.BCEWithLogitsLoss()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.5)
Xt = torch.tensor(X_xor, dtype=torch.float32)
# reshape(-1, 1) : une colonne, comme la sortie du modèle. Sans cela, la diffusion
# (broadcasting) calculerait silencieusement une autre loss.
yt = torch.tensor(y_xor, dtype=torch.float32).reshape(-1, 1)

for epoque in range(4000):
    optimiseur.zero_grad()            # les gradients s'accumulent : on les vide
    logits = modele(Xt)               # forward pass
    perte_t = perte_fn(logits, yt)    # à quel point on se trompe
    perte_t.backward()                # la chain rule, automatiquement
    optimiseur.step()                 # W -= lr * gradient, pour chaque paramètre

print(f"  loss finale  : {perte_t.item():.6f}")
print(f"  probabilités : {torch.sigmoid(modele(Xt))[:, 0].detach().numpy().round(4)}")

In [ ]:
# Le moment de vérité : autograd sur la fiche, en float64 partout (paramètres ET
# cible), contre vos gradients de l'exercice 2.2.
t = {nom: torch.tensor(valeur, dtype=torch.float64, requires_grad=True)
     for nom, valeur in PARAMS.items()}
t_z2 = t["W2"] @ torch.relu(t["W1"] @ torch.tensor(x) + t["b1"]) + t["b2"]
t_perte = nn.functional.binary_cross_entropy_with_logits(
    t_z2, torch.tensor([y_vrai], dtype=torch.float64))
t_perte.backward()
print(f"  loss : à la main {perte_fiche(PARAMS):.12f}   autograd {t_perte.item():.12f}")
for nom, grad in analytiques.items():
    if grad is None:
        print(f"  dL/d{nom:3s} : à compléter (exercice 2.2)")
    else:
        print(f"  dL/d{nom:3s} : écart maximal {np.abs(np.asarray(grad) - t[nom].grad.numpy()).max():.1e}")

In [ ]:
# TODO : reprenez votre training loop PyTorch dans une fonction entrainer_torch(bug)
# et mesurez la loss finale, la loss maximale atteinte et l'accuracy dans trois cas :
#   bug=None             : la training loop correcte ;
#   bug="sans_zero_grad" : la ligne optimiseur.zero_grad() est supprimée ;
#   bug="double_sigmoide": on passe torch.sigmoid(logits) à BCEWithLogitsLoss.
# Avant d'exécuter, écrivez votre prédiction pour chaque cas.
def entrainer_torch(bug=None, epoques=4000):
    torch.manual_seed(SEED)
    m = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
    opt = torch.optim.SGD(m.parameters(), lr=0.5)
    pertes = []
    for _ in range(epoques):
        if bug != "sans_zero_grad":
            opt.zero_grad()
        sortie = m(Xt)
        if bug == "double_sigmoide":
            sortie = torch.sigmoid(sortie)   # BCEWithLogitsLoss en rajoutera une
        perte_b = perte_fn(sortie, yt)
        perte_b.backward()
        opt.step()
        pertes.append(perte_b.item())
    with torch.no_grad():
        logit = m(Xt)
        proba = torch.sigmoid(torch.sigmoid(logit) if bug == "double_sigmoide" else logit)
    exact = ((proba[:, 0] > 0.5).float() == yt[:, 0]).float().mean().item()
    return {"cas": bug or "correct", "loss finale": round(pertes[-1], 4),
            "loss max": round(max(pertes), 4), "accuracy": exact}


print(pd.DataFrame([entrainer_torch(b) for b in (None, "sans_zero_grad", "double_sigmoide")])
      .to_string(index=False))

### Exercice 2.6 · Learning rate et dead units

In [ ]:
# TODO (1) : vos prédictions, par écrit, AVANT d'exécuter.
mes_predictions = {0.001: "____", 0.05: "____", 0.5: "____", 5.0: "____", 50.0: "____"}

# TODO (2) : pour chaque learning rate, entraînez ReseauNumPy() 4 000 epochs et relevez :
#   - la loss finale ;
#   - le nombre de dead units : unités dont la sortie A1 est nulle pour les 4 points
#     (après reseau.forward(X_xor), compter (reseau.A1.max(axis=0) == 0).sum()) ;
#   - un verdict : "diverge", "unités détruites", "rampe encore" ou "converge".

In [ ]:
lignes = []
for taux in (0.001, 0.05, 0.5, 5.0, 50.0):
    r = ReseauNumPy()
    h = entrainer(r, X_xor, y_xor, taux=taux)
    r.forward(X_xor)
    mortes = int((r.A1.max(axis=0) == 0).sum())
    finale = float(h[-1])
    # Le diagnostic repose sur deux mesures, pas une : la loss seule ne distingue pas
    # « trop lent » de « capacité détruite ».
    if not np.isfinite(finale) or finale > h[0]:
        verdict = "diverge"
    elif mortes >= 3:
        verdict = "unités détruites"
    elif finale > 0.35:
        verdict = "rampe encore"
    else:
        verdict = "converge"
    lignes.append({"learning rate": taux, "loss finale": round(finale, 6),
                   "dead units": mortes, "verdict": verdict})
print(pd.DataFrame(lignes).to_string(index=False))

In [ ]:
# TODO :
#  (a) remède 1 : quel learning rate choisir ? Vérifiez.
#  (b) remède 2 : créez ReseauLeaky(ReseauNumPy) qui remplace activation et
#      d_activation par Leaky ReLU (alpha = 0,1), puis entraînez-le à 5,0 et à 50,0.
#      Comptez les unités « éteintes » : (reseau.Z1.max(axis=0) <= 0).sum().
class ReseauLeaky(ReseauNumPy):
    """Même réseau, mais une unité éteinte garde une pente alpha : elle peut revenir."""

    alpha = 0.1

    def activation(self, Z):
        return np.where(Z > 0, Z, self.alpha * Z)

    def d_activation(self, Z):
        return np.where(Z > 0, 1.0, self.alpha)


for nom, classe, taux in [("ReLU, learning rate 0,5", ReseauNumPy, 0.5),
                          ("Leaky ReLU, learning rate 5", ReseauLeaky, 5.0),
                          ("Leaky ReLU, learning rate 50", ReseauLeaky, 50.0)]:
    r = classe()
    with np.errstate(over="ignore", invalid="ignore"):
        h = entrainer(r, X_xor, y_xor, taux=taux)
        r.forward(X_xor)
    exact = ((r.A2[:, 0] > 0.5) == y_xor).mean()
    eteintes = int((r.Z1.max(axis=0) <= 0).sum())
    print(f"  {nom:28s} loss finale {h[-1]:.6f}   accuracy {exact:.2f}   "
          f"unités éteintes {eteintes}")

### TD 3 · Entraîner en pratique : le pipeline PyTorch sur données réelles

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host
from src.training import (EarlyStopping, TabularDataset, evaluate, fit,
                          loaders_from_arrays, make_mlp)

set_seed(SEED)
torch.manual_seed(SEED)
# Quatre threads, pour deux raisons. (1) Sur des réseaux aussi petits, davantage de
# threads RALENTIT l'entraînement : le coût de coordination dépasse le gain. (2) Le
# nombre de threads change l'ordre des additions en virgule flottante, donc les
# derniers chiffres des résultats : les valeurs de référence des séances 9 et 10 ont
# été obtenues avec 4 threads. Même seed, autre nombre de threads : autres décimales.
torch.set_num_threads(4)

# Les mêmes données, la même cible et les mêmes folds qu'en séances 7 et 9.
df = load_raw()
train, _ = split_by_host(df)
train["licensed"] = train["license"].map(
    lambda s: bool(re.search(r"HUT|HB-|AJ0|ESFC", str(s))) if str(s) != "nan" else False)
NUMERIC = ["accommodates", "bedrooms", "beds", "bathrooms", "latitude", "longitude",
           "minimum_nights", "number_of_reviews", "review_scores_rating",
           "calculated_host_listings_count", "availability_365"]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]
X = train[NUMERIC + CATEGORICAL]
y = train["licensed"].astype(int)
FOLDS = list(GroupKFold(n_splits=5).split(X, y, groups=train["host_id"]))
train_idx, val_idx = FOLDS[0]
print(f"annonces {len(train):,}   part de licences {y.mean():.4f}   torch {torch.__version__}")

*Exercice 3.1* · Tenseurs et autograd

In [ ]:
a = torch.tensor([1.0, 2.0])                       # créé par PyTorch : float32
b = torch.tensor(np.array([1.0, 2.0]))             # venu de NumPy : garde float64
print(f"  types : {a.dtype} contre {b.dtype}")

prediction = torch.zeros(4, 1)                     # sortie d'un réseau : (n, 1)
cible = torch.tensor([1.0, 0.0, 1.0, 0.0])         # une cible mal formée : (n,)
print(f"  (4, 1) - (4,) donne une forme {tuple((prediction - cible).shape)} : "
      "le broadcasting a fabriqué une matrice")
print(f"  avec la cible en colonne : {tuple((prediction - cible.reshape(-1, 1)).shape)}")

In [ ]:
# TODO : pour L(w, b) = moyenne de (w*x + b - y)^2, avec x = [1, 2, 3, 4],
# y = [2, 4, 6, 8], w = 0,5 et b = 0 :
#  (1) créez w et b avec requires_grad=True, calculez L, appelez L.backward() ;
#  (2) comparez w.grad et b.grad aux formules dL/dw = moyenne(2 (w x + b - y) x) et
#      dL/db = moyenne(2 (w x + b - y)).
xs = torch.tensor([1.0, 2.0, 3.0, 4.0])
ys = torch.tensor([2.0, 4.0, 6.0, 8.0])
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
L = ((w * xs + b - ys) ** 2).mean()
L.backward()                         # parcourt le graphe enregistré, à rebours
residu = (0.5 * xs + 0.0 - ys).detach()
grad_autograd = [w.grad.item(), b.grad.item()]
grad_formule = [(2 * residu * xs).mean().item(), (2 * residu).mean().item()]
print(f"  autograd : dL/dw = {grad_autograd[0]:.4f}   dL/db = {grad_autograd[1]:.4f}")
print(f"  formule  : dL/dw = {grad_formule[0]:.4f}   dL/db = {grad_formule[1]:.4f}")

In [ ]:
if grad_autograd is None:
    print("  à compléter")
else:
    ok = np.allclose(grad_autograd, grad_formule, atol=1e-6)
    print(f"  {'OK' if ok else 'à revoir'}")
#On a applique la rule chain

**Exercice 3.2** · Préparer les données : le fold et le feature scaling

In [ ]:
brut = X.iloc[train_idx][NUMERIC]
print(brut.describe().loc[["min", "max"]].T.round(2).to_string())

In [2]:
# TODO : construisez le préprocesseur de la séance 9 (médiane + StandardScaler pour
# NUMERIC ; mode + OneHotEncoder(handle_unknown="ignore", min_frequency=20,
# sparse_output=False) pour CATEGORICAL), ajustez-le sur X.iloc[train_idx] SEULEMENT,
# puis transformez les deux parties du fold.
# Affichez les formes, et le nombre de colonnes indicatrices.

Le principe du OneHotEncoder: on une colonne qu'on essaye de transformer en n colonnes. On fait un check. C'est la trans formation numérique des colonnes

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                   min_frequency=20,
                                                   sparse_output=False))]), CATEGORICAL),
    ])


prep = make_preprocessor().fit(X.iloc[train_idx])     # entraînement du fold SEULEMENT
X_tr = prep.transform(X.iloc[train_idx])
X_va = prep.transform(X.iloc[val_idx])                 # transformation seule
y_tr = y.iloc[train_idx].to_numpy()
y_va = y.iloc[val_idx].to_numpy()
print(f"  {X.shape[1]} colonnes d'origine -> {X_tr.shape[1]} features")
print(f"  entraînement {X_tr.shape}   validation {X_va.shape}")
print(f"  dont {X_tr.shape[1] - len(NUMERIC)} colonnes indicatrices")

In [3]:
# TODO : pour chaque feature de NUMERIC, calculez le gradient, la courbure et le learning
# rate stable maximal 2 / courbure :
#   (a) sur les données imputées mais SANS scaling
#       (SimpleImputer(strategy="median") ajusté sur l'entraînement) ;
#   (b) sur les 11 premières colonnes de X_tr (standardisées).
# Affichez un tableau, puis le rapport entre le plus grand et le plus petit learning rate
# stable dans chaque cas.

In [ ]:
imputeur = SimpleImputer(strategy="median").fit(brut)
X_brut = imputeur.transform(brut)
X_std = X_tr[:, :len(NUMERIC)]
residu = 0.5 - y_tr
tableau = pd.DataFrame({
    "gradient brut": np.abs((residu[:, None] * X_brut).mean(axis=0)),
    "learning rate stable brut": 2 / (0.25 * (X_brut ** 2).mean(axis=0)),
    "gradient standardisé": np.abs((residu[:, None] * X_std).mean(axis=0)),
    "learning rate stable standardisé": 2 / (0.25 * (X_std ** 2).mean(axis=0)),
}, index=NUMERIC)
print(tableau.to_string(float_format=lambda v: f"{v:.2e}"))
for cas in ("brut", "standardisé"):
    t = tableau[f"learning rate stable {cas}"]
    print(f"\n  {cas:12s} : learning rate stable de {t.min():.1e} à {t.max():.1e}, "
          f"rapport {t.max() / t.min():,.0f}")

**Exercice 3.3** · Dataset et DataLoader

Un batch c'est un exemple qu'on va utiliser pour un...

In [ ]:
# TODO : complétez MonDataset. Les X en float32 ; la cible en float32 et en COLONNE.
class MonDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(np.asarray(X, dtype=np.float32))
        # En colonne : (n, 1), comme la sortie du réseau (exercice 3.1).
        self.y = torch.as_tensor(np.asarray(y, dtype=np.float32)).reshape(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

In [ ]:
# Vérification contre le TabularDataset du module src/training.py.
mien, reference = MonDataset(X_tr, y_tr), TabularDataset(X_tr, y_tr)
ok = (len(mien) == len(reference) and torch.equal(mien[0][0], reference[0][0])
      and torch.equal(mien[0][1], reference[0][1]))
print(f"  {len(mien)} exemples, premier exemple identique : {'OK' if ok else 'à revoir'}")

In [4]:
# TODO : pour des batches de 64, 256 et 1024 (DataLoader(..., shuffle=True)), affichez
# le nombre de batches par epoch (len(loader)), la forme du premier batch, et le nombre
# de mises à jour pour 20 epochs.

In [ ]:
#L'impact du batch dans les MAJ plus de batch on a moins de maj a faire eet le contraire
for taille in (64, 256, 1024):
    chargeur = DataLoader(MonDataset(X_tr, y_tr), batch_size=taille, shuffle=True)
    xb, yb = next(iter(chargeur))
    print(f"  batches de {taille:4d} : {len(chargeur):3d} batches par epoch "
          f"(= ceil({len(X_tr)} / {taille}) = {math.ceil(len(X_tr) / taille)}), premier batch "
          f"X {tuple(xb.shape)} y {tuple(yb.shape)}, {20 * len(chargeur):5d} mises à jour en 20 epochs")

**Exercice 3.4** · La training loop

In [ ]:
# TODO : écrivez la training loop, 25 epochs, et remplissez historique["train"] et
# historique["val"] à chaque epoch.
torch.manual_seed(SEED)
modele = make_mlp(X_tr.shape[1], hidden=(64, 32))
perte_fn = nn.BCEWithLogitsLoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3)
chargeur_tr, chargeur_va = loaders_from_arrays(X_tr, y_tr, X_va, y_va,
                                               batch_size=256, seed=SEED)
historique = {"train": [], "val": []}
for epoque in range(25):
    modele.train()
    cumul, vus = 0.0, 0
    for xb, yb in chargeur_tr:
        optimiseur.zero_grad()
        perte = perte_fn(modele(xb), yb)
        perte.backward()
        optimiseur.step()
        # .item() : un nombre Python, détaché du graphe (sinon fuite de mémoire).
        cumul += perte.item() * len(xb)
        vus += len(xb)
    historique["train"].append(cumul / vus)
#pour historique on a un dictionnaire avec des listes par paire
#une liste de valeur de train et une liste de validation
    modele.eval()
    with torch.no_grad():
        total, vus_v = 0.0, 0
        for xb, yb in chargeur_va:
            total += perte_fn(modele(xb), yb).item() * len(xb)
            vus_v += len(xb)
    historique["val"].append(total / vus_v)

In [ ]:
# Vérification : la séance 9 avait trouvé un minimum de validation de 0,4424 à
# l'epoch 8, avec exactement ces réglages.
if not historique["val"]:
    print("  à compléter")
else:
    meilleure = int(np.argmin(historique["val"]))
    print(f"  epoch 1 : entraînement {historique['train'][0]:.4f}  validation {historique['val'][0]:.4f}")
    print(f"  minimum de validation {historique['val'][meilleure]:.4f} à l'epoch {meilleure + 1}")
    ok = abs(historique["val"][meilleure] - 0.4424) < 1e-3 and meilleure + 1 == 8
    print(f"  identique à la séance 9 : {'OK' if ok else 'à revoir'}")
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(range(1, 26), historique["train"], label="entraînement")
    ax.plot(range(1, 26), historique["val"], label="validation")
    ax.axvline(meilleure + 1, color="grey", ls=":", label=f"minimum ({meilleure + 1})")
    ax.set(xlabel="epoch", ylabel="cross-entropy", title="Suivre la validation dès l'epoch 1")
    ax.legend()
    plt.tight_layout()
    plt.show()
#Les pointillets nous montrent le point de convergence des erreurs

In [ ]:
# Votre boucle contre fit() de src/training.py : même seed, mêmes chargeurs.
torch.manual_seed(SEED)
modele_fit = make_mlp(X_tr.shape[1], hidden=(64, 32))
tr_f, va_f = loaders_from_arrays(X_tr, y_tr, X_va, y_va, batch_size=256, seed=SEED)
historique_fit = fit(modele_fit, tr_f, va_f, epochs=25, lr=1e-3, seed=SEED)
if historique["val"]:
    ecart = np.abs(np.array(historique["val"]) - np.array(historique_fit["val"])).max()
    print(f"  écart maximal entre votre boucle et fit() : {ecart:.1e}")

**Exercice 3.5** · Optimizers : SGD, momentum, Adam

In [5]:
# TODO : pour chacune des configurations ci-dessous, entraînez make_mlp(77, (64, 32))
# pendant 15 epochs (batches de 256, seed SEED, fit() avec optimizer=...), puis relevez
# la training loss aux epochs 1, 5 et 15, le minimum de validation et l'AUC.
#   "SGD learning rate 0,01"     : torch.optim.SGD(params, lr=0.01)
#   "SGD learning rate 0,1"      : torch.optim.SGD(params, lr=0.1)
#   "SGD + momentum 0,9, 0,01"   : torch.optim.SGD(params, lr=0.01, momentum=0.9)
#   "Adam learning rate 0,001"   : torch.optim.Adam(params, lr=1e-3)

In [ ]:
CONFIGS_OPT = {
    "SGD learning rate 0,01": lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD learning rate 0,1": lambda p: torch.optim.SGD(p, lr=0.1),
    "SGD + momentum 0,9, 0,01": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam learning rate 0,001": lambda p: torch.optim.Adam(p, lr=1e-3),
}
lignes, courbes = [], {}
for nom, fabrique in CONFIGS_OPT.items():
    torch.manual_seed(SEED)
    net = make_mlp(X_tr.shape[1], hidden=(64, 32))
    tr_l, va_l = loaders_from_arrays(X_tr, y_tr, X_va, y_va, batch_size=256, seed=SEED)
    h = fit(net, tr_l, va_l, epochs=15, optimizer=fabrique(net.parameters()), seed=SEED)
    _, logits = evaluate(net, va_l, nn.BCEWithLogitsLoss())
    courbes[nom] = h["train"]
    lignes.append({"optimizer": nom, "train epoch 1": round(h["train"][0], 4),
                   "train epoch 5": round(h["train"][4], 4), "train epoch 15": round(h["train"][-1], 4),
                   "val min": round(min(h["val"]), 4), "AUC": round(roc_auc_score(y_va, logits.ravel()), 4)})
print(pd.DataFrame(lignes).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.6))
for nom, c in courbes.items():
    ax.plot(range(1, 16), c, marker="o", ms=3, label=nom)
ax.set(xlabel="epoch", ylabel="training loss", title="Même réseau, quatre optimizers")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
#L'optimiseur adam est meilleur

**Exercice 3.6** · Planifier le learning rate

In [7]:
# TODO : écrivez entrainer_planifie(fabrique_planif) qui entraîne make_mlp(77, (64, 32))
# 20 epochs avec SGD(lr=0.05, momentum=0.9), batches de 256, et appelle planif.step()
# à la fin de chaque epoch. Comparez :
#   "constant"    : aucun schedule ;
#   "paliers"     : torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.1) ;
#   "cosinus"     : torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20).
# Relevez la validation loss finale, son minimum, et l'AUC finale.

In [ ]:
def entrainer_planifie(fabrique_planif, epoques=20, graine=SEED):
    torch.manual_seed(graine)
    net = make_mlp(X_tr.shape[1], hidden=(64, 32))
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9)
    planif = fabrique_planif(opt) if fabrique_planif else None
    tr_l, va_l = loaders_from_arrays(X_tr, y_tr, X_va, y_va, batch_size=256, seed=graine)
    lf = nn.BCEWithLogitsLoss()
    val, taux = [], []
    for _ in range(epoques):
        net.train()
        for xb, yb in tr_l:
            opt.zero_grad()
            lf(net(xb), yb).backward()
            opt.step()
        taux.append(opt.param_groups[0]["lr"])
        if planif:
            planif.step()                     # une fois par epoch, après l'epoch
        v, logits = evaluate(net, va_l, lf)
        val.append(v)
    return val, taux, roc_auc_score(y_va, logits.ravel())


PLANIFS = {"constant": None,
           "paliers": lambda o: torch.optim.lr_scheduler.StepLR(o, step_size=10, gamma=0.1),
           "cosinus": lambda o: torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=20)}
lignes = []
for nom, fabrique in PLANIFS.items():
    val, taux, auc = entrainer_planifie(fabrique)
    lignes.append({"schedule": nom, "learning rate final": f"{taux[-1]:.5f}",
                   "val finale": round(val[-1], 4), "val min": round(min(val), 4),
                   "AUC finale": round(auc, 4)})
print(pd.DataFrame(lignes).to_string(index=False))

**Exercice 3.7** · L'initialisation

In [9]:
# TODO : écrivez initialiser(net, mode) qui parcourt les nn.Linear du réseau et applique :
#   "défaut PyTorch" : rien ;
#   "zéros"          : nn.init.zeros_ sur poids et biais ;
#   "constante 0,1"  : nn.init.constant_(poids, 0.1), biais à zéro ;
#   "normale σ = 3"  : nn.init.normal_(poids, 0, 3.0) ;
#   "He"             : nn.init.kaiming_normal_(poids, nonlinearity="relu"), biais à zéro.
# Puis entraînez chaque variante 15 epochs (batches de 512, Adam 1e-3) et relevez l'AUC,
# la training loss de l'epoch 1 et le nombre d'unités distinctes de la première
# couche : len(np.unique(net[0].weight.detach().numpy().round(6), axis=0)).

In [ ]:
def initialiser(net, mode):
    for couche in net:
        if not isinstance(couche, nn.Linear):
            continue
        if mode == "zéros":
            nn.init.zeros_(couche.weight)
            nn.init.zeros_(couche.bias)
        elif mode == "constante 0,1":
            nn.init.constant_(couche.weight, 0.1)
            nn.init.zeros_(couche.bias)
        elif mode == "normale σ = 3":
            nn.init.normal_(couche.weight, 0.0, 3.0)
        elif mode == "He":
            nn.init.kaiming_normal_(couche.weight, nonlinearity="relu")
            nn.init.zeros_(couche.bias)


lignes = []
for mode in ("défaut PyTorch", "zéros", "constante 0,1", "normale σ = 3", "He"):
    torch.manual_seed(SEED)
    net = make_mlp(X_tr.shape[1], hidden=(64, 32))
    initialiser(net, mode)
    tr_l, va_l = loaders_from_arrays(X_tr, y_tr, X_va, y_va, batch_size=512, seed=SEED)
    h = fit(net, tr_l, va_l, epochs=15, lr=1e-3, seed=SEED)
    _, logits = evaluate(net, va_l, nn.BCEWithLogitsLoss())
    distinctes = len(np.unique(net[0].weight.detach().numpy().round(6), axis=0))
    lignes.append({"initialisation": mode, "AUC": round(roc_auc_score(y_va, logits.ravel()), 4),
                   "train epoch 1": round(h["train"][0], 4), "train epoch 15": round(h["train"][-1], 4),
                   "unités distinctes (couche 1)": f"{distinctes}/64"})
print(pd.DataFrame(lignes).to_string(index=False))